
# Binary Classification — Предсказание задержек рейсов

**Цель:** показать бинарную классификацию с разными типами признаков.  
**Данные:** синтетика по рейсам: `airline`, `month`, `distance`, `weather_index`, `departure_hour` → `delayed` (0/1).  
**Модель:** `RandomForestClassifier`.  
**Метрики:** accuracy, precision, recall, ROC-AUC.

> Самодостаточный ноут с обсуждением важностей признаков.


In [ ]:
# Импорты
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, RocCurveDisplay

np.random.seed(7)
print("Импорты готовы")


## 1. Генерация синтетических данных


In [ ]:
n = 800
airlines = np.random.choice(["AL1","AL2","AL3","AL4"], size=n, p=[0.3,0.3,0.25,0.15])
month = np.random.randint(1, 13, size=n)
distance = np.random.randint(100, 3500, size=n)
weather_index = np.random.choice([0,1,2], size=n, p=[0.7,0.25,0.05])  # 0=ок,1=дождь/ветер,2=шторм
departure_hour = np.random.randint(0, 24, size=n)

# Базовая вероятность задержки
base = 0.1 + 0.15*(weather_index==1) + 0.35*(weather_index==2)
base += 0.05*((departure_hour>=20) | (departure_hour<=6))  # ночью чаще задержки
base += 0.00005*distance  # дальние рейсы чаще двигаются
base += np.where(airlines=="AL4", 0.05, 0)  # условно более проблемная AL4

proba = np.clip(base, 0, 0.95)
delayed = (np.random.rand(n) < proba).astype(int)

df = pd.DataFrame({
    "airline": airlines,
    "month": month,
    "distance": distance,
    "weather_index": weather_index,
    "departure_hour": departure_hour,
    "delayed": delayed
})

df.head()


## 2. Кодирование и разбиение


In [ ]:
df_enc = pd.get_dummies(df, columns=["airline"], drop_first=True)

X = df_enc.drop(columns=["delayed"])
y = df_enc["delayed"]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7, stratify=y
)

X_train.shape, X_test.shape


## 3. Модель и метрики


In [ ]:
rf = RandomForestClassifier(n_estimators=300, random_state=7)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:,1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_proba)

print(f"Accuracy: {acc:.3f} | Precision: {prec:.3f} | Recall: {rec:.3f} | F1: {f1:.3f} | ROC-AUC: {roc:.3f}")

In [ ]:
# ROC-кривая
fig, ax = plt.subplots(figsize=(6,4))
RocCurveDisplay.from_predictions(y_test, y_proba, ax=ax)
ax.set_title("ROC-кривая (RandomForest)")
plt.show()


## 4. Важности признаков


In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Топ признаков:")
display(importances.head(10))

In [ ]:
plt.figure(figsize=(6,4))
importances.head(12).iloc[::-1].plot(kind="barh")
plt.title("Feature Importances (Top 12)")
plt.tight_layout()
plt.show()


## 5. Вопросы для обсуждения
1) Какие признаки оказались ключевыми и почему? Совпадает ли это с интуицией?  
2) Что добавили бы в качестве фич: день недели, аэропорт, стыковки, сезонные эффекты?  
3) Для бизнеса важнее точность или полнота в этой задаче? Почему?
